# Sprint 1: Infraestructura de Datos y Pipeline de Calibracion

> **Prerrequisitos:** Solo requiere ejecutar el **Bloque 0** (configuracion de entorno y parches).
> Este sprint es el punto de entrada del proyecto.

**Objetivo:** Consolidar el entorno de datos local en WSL2 y construir el cargador de tensores
que alimentara los algoritmos de cuantizacion offline (GPTQ y AWQ) en la RTX 4060 Ti.

**Dataset:** [fsicoli/common_voice_17_0](https://huggingface.co/datasets/fsicoli/common_voice_17_0) (config `es`)
- Mirror comunitario completo de Mozilla Common Voice 17.0 Spanish (~2,220h total, 562h validadas)
- Splits nativos: `train`, `validation`, `test`, `other`, `invalidated`
- Audio original a 16kHz mono, licencia CC0-1.0
- **Motivo del cambio:** Mozilla retiro CV 17.0 de HuggingFace en Oct 2025. El mirror `fsicoli` almacena los audios directamente.

> **Nota tecnica:** VibeVoice 1.5B procesa audio crudo a **24kHz mono**, no espectrogramas de Mel.
> El preprocesamiento consiste en remuestreo + normalizacion de amplitud, no en conversion frecuencial.

---
## Bloque 0: Configuracion del Entorno y Parches Criticos

Se aplica el parche obligatorio de `transformers` para evitar colisiones de mapeo
con el fork comunitario de VibeVoice (ver `agents.md` seccion 3).

In [3]:
import sys
import os
import json
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import transformers
from datasets import load_dataset, Audio, Dataset, DatasetDict, concatenate_datasets
from tqdm.auto import tqdm

# --- Parche critico: colision de mapeo en transformers ---
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)
print("Parche de CONFIG_MAPPING aplicado.")

# --- Rutas del proyecto ---
PROJECT_ROOT = Path(os.path.abspath("")).parent if "notebooks" in os.path.abspath("") else Path(os.path.abspath(""))
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"WEIGHTS_DIR  : {WEIGHTS_DIR}")
print(f"Torch        : {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Parche de CONFIG_MAPPING aplicado.
PROJECT_ROOT : /home/alfrog/projects/VibeVoice_Optimization
DATA_DIR     : /home/alfrog/projects/VibeVoice_Optimization/data
WEIGHTS_DIR  : /home/alfrog/projects/VibeVoice_Optimization/weights
Torch        : 2.12.0+cu130  |  CUDA: True
GPU          : NVIDIA GeForce RTX 4060 Ti
VRAM Total   : 8.00 GB


---
## TSK-1.1: Descarga Automatizada del Dataset

Se descarga el subset en espanol de Common Voice 17.0 desde el mirror comunitario
`fsicoli/common_voice_17_0` con config `"es"`. El dataset contiene las particiones
nativas `train`/`validation`/`test`/`other` con audios y transcripciones.

**Estrategia:** Cargamos el split `train` como base para el 70% de datos de
entrenamiento (fine-tuning), y los splits `test` y `validation` nativos como
nuestros sets de benchmark y calibracion respectivamente.

In [4]:
DATASET_ID = "fsicoli/common_voice_17_0"
DATASET_CONFIG = "es"

print(f"Descargando dataset: {DATASET_ID} (config={DATASET_CONFIG})")
print("(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)\n")

t_start = time.time()

cv17_dataset = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="train",
    trust_remote_code=True,
)

t_elapsed = time.time() - t_start
print(f"\nDescarga completada en {t_elapsed/60:.1f} minutos.")
print(f"Registros cargados (train): {len(cv17_dataset):,}")
print(f"Columnas: {cv17_dataset.column_names}")

Descargando dataset: fsicoli/common_voice_17_0 (config=es)
(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)



Reading metadata...: 336846it [00:01, 302211.76it/s]s/s]
Generating train split: 336846 examples [00:40, 8267.73 examples/s] 
Reading metadata...: 15857it [00:00, 328621.65it/s]xamples/s]
Generating validation split: 15857 examples [00:02, 7788.17 examples/s]
Reading metadata...: 15857it [00:00, 350204.45it/s]s/s]
Generating test split: 15857 examples [00:02, 7865.53 examples/s]
Reading metadata...: 1138631it [00:03, 311900.66it/s]/s]
Generating other split: 1138631 examples [02:04, 9145.43 examples/s] 
Reading metadata...: 86227it [00:00, 352991.03it/s]examples/s]
Generating invalidated split: 86227 examples [00:09, 8963.00 examples/s] 



Descarga completada en 28.6 minutos.
Registros cargados (train): 336,846
Columnas: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant']


---
## Exploracion del Dataset

Inspeccionamos la estructura, metadatos demograficos y distribucion de duraciones
para validar la calidad de la muestra antes de la particion.

In [5]:
# --- Muestra de registros ---
print("=== Ejemplos del dataset ===\n")
for i in range(3):
    sample = cv17_dataset[i]
    audio_info = sample["audio"]
    duration_s = len(audio_info["array"]) / audio_info["sampling_rate"]
    print(f"[{i}] \"{sample['sentence'][:80]}...\"")
    print(f"    Duracion: {duration_s:.2f}s | SR: {audio_info['sampling_rate']}Hz")
    print(f"    Locale: {sample['locale']} | Genero: {sample['gender']} | Edad: {sample['age']}")
    print()

# --- Distribucion de duraciones (muestreo estadistico) ---
SAMPLE_SIZE = 5000
indices = random.sample(range(len(cv17_dataset)), min(SAMPLE_SIZE, len(cv17_dataset)))
durations = []
for idx in tqdm(indices, desc="Muestreando duraciones"):
    arr = cv17_dataset[idx]["audio"]["array"]
    sr = cv17_dataset[idx]["audio"]["sampling_rate"]
    durations.append(len(arr) / sr)

durations = np.array(durations)
print(f"\n=== Estadisticas de Duracion (muestra n={len(durations)}) ===")
print(f"Media    : {durations.mean():.2f}s")
print(f"Mediana  : {np.median(durations):.2f}s")
print(f"Min/Max  : {durations.min():.2f}s / {durations.max():.2f}s")
print(f"Std      : {durations.std():.2f}s")
print(f"Horas est. totales: {durations.mean() * len(cv17_dataset) / 3600:.1f}h")

# --- Distribucion de genero y edad ---
genders = Counter()
ages = Counter()
for idx in tqdm(indices, desc="Muestreando demografia"):
    sample = cv17_dataset[idx]
    genders[sample["gender"]] += 1
    ages[sample["age"]] += 1

print(f"\nGeneros  : {dict(genders.most_common())}")
print(f"Edades    : {dict(ages.most_common())}")

=== Ejemplos del dataset ===

[0] "A ese grupo de amigos les gusta comer helados por la tarde...."
    Duracion: 3.89s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[1] "A mi profesora le importa demasiado la gramática y muy poco el contenido...."
    Duracion: 5.16s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[2] "A decir verdad , eres demasiado genial ...."
    Duracion: 3.05s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties



Muestreando duraciones: 100%|██████████| 5000/5000 [00:33<00:00, 148.61it/s]



=== Estadisticas de Duracion (muestra n=5000) ===
Media    : 5.19s
Mediana  : 5.11s
Min/Max  : 1.58s / 10.44s
Std      : 1.44s
Horas est. totales: 485.2h


Muestreando demografia: 100%|██████████| 5000/5000 [00:13<00:00, 360.70it/s]


Generos  : {'male_masculine': 2864, 'female_feminine': 1576, '': 560}
Edades    : {'twenties': 2387, 'thirties': 616, '': 542, 'sixties': 509, 'fifties': 389, 'teens': 292, 'fourties': 262, 'seventies': 1, 'nineties': 1, 'eighties': 1}


---
## TSK-1.2: Particion de la Muestra

Common Voice 17.0 ya proporciona divisiones nativas. Las usamos directamente
para mantener trazabilidad con el estandar de la comunidad:

| Particion | Split CV17 | Proposito en el proyecto |
|-----------|-----------|---------------------------|
| Test      | `test`    | Benchmark final (Sprint 5) |
| Validacion| `validation` + muestra `other` | Calibracion offline (GPTQ/AWQ) |
| Train     | `train`   | Fine-tuning espanol (Sprint 1.5) |

Cargamos todos los splits y guardamos los indices para trazabilidad.

In [6]:
# --- Cargar todos los splits nativos de CV17 es ---
print("Cargando splits nativos de Common Voice 17.0 (es)...")

cv17_all = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    trust_remote_code=True,
)

for split_name, split_data in cv17_all.items():
    print(f"  {split_name:<20} {len(split_data):>10,} registros")

train_dataset = cv17_all["train"]
test_dataset = cv17_all["test"]
val_dataset = cv17_all["validation"]
other_dataset = cv17_all.get("other", None)

print(f"\n=== Asignacion para el proyecto ===")
print(f"Train (fine-tuning)       : {len(train_dataset):,} registros")
print(f"Test (benchmark)          : {len(test_dataset):,} registros")
print(f"Validation (calibracion)  : {len(val_dataset):,} registros")
if other_dataset:
    print(f"Other (calibracion extra) : {len(other_dataset):,} registros")

Cargando splits nativos de Common Voice 17.0 (es)...
  train                   336,846 registros
  validation               15,857 registros
  test                     15,857 registros
  other                 1,138,631 registros
  invalidated              86,227 registros

=== Asignacion para el proyecto ===
Train (fine-tuning)       : 336,846 registros
Test (benchmark)          : 15,857 registros
Validation (calibracion)  : 15,857 registros
Other (calibracion extra) : 1,138,631 registros


In [7]:
# --- Guardar metadatos de particiones para trazabilidad ---
partitions_meta = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "train_count": len(train_dataset),
    "test_count": len(test_dataset),
    "val_count": len(val_dataset),
    "other_count": len(other_dataset) if other_dataset else 0,
    "splits_used": {
        "fine_tuning": "train",
        "benchmark_test": "test",
        "calibration": "validation (+ other)",
    },
}

partitions_path = DATA_DIR / "dataset_partitions.json"
with open(partitions_path, "w") as f:
    json.dump(partitions_meta, f, indent=2)
print(f"Metadatos de particiones guardados en: {partitions_path}")

Metadatos de particiones guardados en: /home/alfrog/projects/VibeVoice_Optimization/data/dataset_partitions.json


---
## TSK-1.3: Preprocesamiento Acustico

VibeVoice 1.5B espera audio crudo monofonico a **24,000 Hz** (no espectrogramas de Mel).
El pipeline de preprocesamiento consiste en:

1. **Remuestreo:** 48,000 Hz → 24,000 Hz (usando `librosa.resample`)
2. **Conversion a mono:** garantizado por el dataset
3. **Normalizacion de amplitud:** a -25 dB FS (mismo criterio que `AudioNormalizer` del modelo)
4. **Validacion:** descarte de audios corruptos o de duracion minima (<0.5s)

El resultado se almacena como tensores `torch.float32` en archivos `.pt`.

In [8]:
import librosa

TARGET_SR = 24000
TARGET_DB_FS = -25.0
MIN_DURATION_S = 0.5
EPS = 1e-6

def preprocess_audio(audio_array: np.ndarray, orig_sr: int) -> np.ndarray | None:
    """
    Preprocesa un array de audio para VibeVoice:
    remuestreo a 24kHz + normalizacion a target_dB_FS.
    Retorna None si el audio es invalido.
    """
    if audio_array is None or len(audio_array) == 0:
        return None

    # Remuestreo a 24kHz
    if orig_sr != TARGET_SR:
        audio = librosa.resample(
            y=audio_array.astype(np.float32),
            orig_sr=orig_sr,
            target_sr=TARGET_SR,
        )
    else:
        audio = audio_array.astype(np.float32)

    duration = len(audio) / TARGET_SR
    if duration < MIN_DURATION_S:
        return None

    # Normalizacion a target dB FS
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < EPS:
        return None
    scalar = (10 ** (TARGET_DB_FS / 20)) / (rms + EPS)
    audio = audio * scalar

    # Prevenir clipping
    peak = np.max(np.abs(audio))
    if peak > 1.0:
        audio = audio / peak

    return audio.astype(np.float32)

# --- Prueba con los primeros 5 registros ---
print("Validando preprocesamiento con 5 muestras...\n")
for i in range(5):
    sample = val_dataset[i]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    processed = preprocess_audio(arr, sr)
    if processed is not None:
        print(f"[{i}] OK: {len(processed)/TARGET_SR:.2f}s -> {processed.shape}, "
              f"peak={np.max(np.abs(processed)):.3f}, rms={np.sqrt(np.mean(processed**2)):.4f}")
    else:
        print(f"[{i}] DESCARTADO")

Validando preprocesamiento con 5 muestras...

[0] OK: 4.85s -> (116352,), peak=0.409, rms=0.0562
[1] OK: 5.57s -> (133632,), peak=0.453, rms=0.0562
[2] OK: 4.37s -> (104832,), peak=0.368, rms=0.0562
[3] OK: 5.18s -> (124416,), peak=0.358, rms=0.0562
[4] OK: 3.34s -> (80064,), peak=0.557, rms=0.0562


---
## TSK-1.4: Construccion del Set de Calibracion Offline

Se construye el tensor de calibracion a partir de la particion de validacion (10%).

**Estrategia:**
- Para GPTQ/AWQ se necesita un subconjunto representativo (tipicamente 128-512 muestras)
  que cubra diversidad de duraciones, generos y acentos.
- Se seleccionan `N_CALIB_SAMPLES` muestras con muestreo estratificado simple.
- Los audios preprocesados se concatenan en un unico tensor `[total_samples]`.
- Se guarda metadato con offsets para reconstruir limites de cada alocucion.

**Formato de salida:**
- `data/calibration_tensor.pt` — tensor float32 con audio concatenado
- `data/calibration_metadata.json` — indices originales, offsets, duraciones

In [9]:
N_CALIB_SAMPLES = 512
CALIB_SEED = 123

random.seed(CALIB_SEED)
np.random.seed(CALIB_SEED)

print(f"Seleccionando {N_CALIB_SAMPLES} muestras de calibracion...")

# --- Usar split validation como base para calibracion ---
calib_source = val_dataset
n_source = len(calib_source)
calib_local_indices = sorted(random.sample(range(n_source), min(N_CALIB_SAMPLES, n_source)))

# --- Preprocesar y recolectar ---
calib_audios = []
calib_metadata = []
rejected = 0
current_offset = 0

for local_idx in tqdm(calib_local_indices, desc="Preprocesando calibracion"):
    sample = calib_source[local_idx]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    processed = preprocess_audio(arr, sr)
    if processed is None:
        rejected += 1
        continue

    n_samples = len(processed)
    calib_audios.append(torch.from_numpy(processed))
    calib_metadata.append({
        "local_idx": local_idx,
        "sentence": sample["sentence"],
        "offset_start": current_offset,
        "offset_end": current_offset + n_samples,
        "n_samples": n_samples,
        "duration_s": n_samples / TARGET_SR,
        "gender": sample.get("gender", ""),
        "age": sample.get("age", ""),
    })
    current_offset += n_samples

print(f"\nMuestras procesadas : {len(calib_audios)}")
print(f"Muestras rechazadas : {rejected}")
print(f"Total de muestras de audio : {current_offset:,} ({current_offset/TARGET_SR:.1f}s)")

Seleccionando 512 muestras de calibracion...


Preprocesando calibracion: 100%|██████████| 512/512 [00:02<00:00, 190.64it/s]


Muestras procesadas : 512
Muestras rechazadas : 0
Total de muestras de audio : 74,889,543 (3120.4s)


In [10]:
# --- Concatenar en tensor unificado y guardar ---
calibration_tensor = torch.cat(calib_audios, dim=0)

tensor_path = DATA_DIR / "calibration_tensor.pt"
torch.save(calibration_tensor, tensor_path)

metadata_path = DATA_DIR / "calibration_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset_id": DATASET_ID,
        "dataset_config": DATASET_CONFIG,
        "split_origin": "validation",
        "target_sr": TARGET_SR,
        "target_db_fs": TARGET_DB_FS,
        "n_samples": len(calib_audios),
        "total_audio_samples": calibration_tensor.shape[0],
        "total_duration_s": calibration_tensor.shape[0] / TARGET_SR,
        "records": calib_metadata,
    }, f, indent=2, ensure_ascii=False)

size_mb = tensor_path.stat().st_size / (1024 * 1024)
print(f"Tensor de calibracion : {tensor_path}")
print(f"Shape                 : {calibration_tensor.shape}")
print(f"Tamaño en disco       : {size_mb:.1f} MB")
print(f"Metadatos             : {metadata_path}")

Tensor de calibracion : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_tensor.pt
Shape                 : torch.Size([74889543])
Tamaño en disco       : 285.7 MB
Metadatos             : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_metadata.json


In [11]:
# --- Verificacion de carga ---
print("=== Verificacion del Set de Calibracion ===\n")

loaded_tensor = torch.load(DATA_DIR / "calibration_tensor.pt", weights_only=True)
with open(DATA_DIR / "calibration_metadata.json", "r") as f:
    loaded_meta = json.load(f)

print(f"Tensor cargado  : {loaded_tensor.shape}, dtype={loaded_tensor.dtype}")
print(f"Duracion total  : {loaded_meta['total_duration_s']:.1f}s")
print(f"Numero de clips : {loaded_meta['n_samples']}")
print(f"\nPrimeros 3 clips:")
for rec in loaded_meta["records"][:3]:
    print(f"  idx={rec['local_idx']:>6} | {rec['duration_s']:.2f}s | "
          f"\"{rec['sentence'][:60]}...\"")

# --- Verificar offsets ---
for i, rec in enumerate(loaded_meta["records"]):
    start, end = rec["offset_start"], rec["offset_end"]
    segment = loaded_tensor[start:end]
    assert len(segment) == rec["n_samples"], f"Offset mismatch en indice {i}"

print("\nVerificacion de offsets: OK")

=== Verificacion del Set de Calibracion ===

Tensor cargado  : torch.Size([74889543]), dtype=torch.float32
Duracion total  : 3120.4s
Numero de clips : 512

Primeros 3 clips:
  idx=     5 | 3.14s | "No lo he visto ...."
  idx=    27 | 5.14s | "Allá va el niño, donde le tratan con cariño...."
  idx=    97 | 6.82s | "de la mano , dando vueltas en setenta y cinco metros de eslo..."

Verificacion de offsets: OK


---
# Sprint 1.5: Fine-Tuning Monolingue Espanol con LoRA

> **Prerrequisitos:** Requiere haber ejecutado **Sprint 1 completo** (celdas Bloque 0 hasta TSK-1.4).
> Variables necesarias: `cv17_all`, `train_dataset`, `val_dataset`, `DATA_DIR`, `WEIGHTS_DIR`, `PROJECT_ROOT`.

**Objetivo:** Adaptar VibeVoice 1.5B a espanol exclusivamente mediante LoRA fine-tuning
sobre el split `train` de Common Voice 17.0. El modelo resultante (**VibeVoice-ES**)
sera la nueva linea base FP16 para todos los experimentos de cuantizacion.

**Estrategia:**
- LoRA rank=8 sobre proyecciones Q/K/V/O + gate/up/down del LLM (Qwen2.5)
- Entrenamiento completo del *diffusion head* para adaptar prosodia espanola
- Tokenizadores acustico y semantico congelados (lenguaje-agnosticos)
- `voice_prompt_drop_rate=1.0` para priorizar sintesis sobre clonacion
- `ce_loss_weight=1.0` (corregido de 0.04) para precision textual
- `diffusion_loss_weight=1.0` balanceado con CE
- `per_device_train_batch_size=1`, `gradient_accumulation_steps=64` para 8GB VRAM

> **Nota:** El entrenamiento ejecuta en terminal via `python -m vibevoice.finetune.train_vibevoice`.
> Las celdas de este bloque preparan los datos y generan el script de lanzamiento.
> La ejecucion del entrenamiento (~7-15h) se realiza fuera del notebook.

---
## TSK-1.5.1: Formatear Dataset al Esquema VibeVoice

El script de fine-tuning espera:
- Columna `text` con formato `"Speaker 1: {sentence}"`
- Columna `audio` como path a archivo de audio (str) o dict con array+sampling_rate
- Columna `voice_prompts` opcional (se auto-genera si no existe)

Guardamos los datos formateados como JSONL para pasarlos con `--train_jsonl` / `--validation_jsonl`.
Los paths de audio apuntan a la cache local de HuggingFace (archivos MP3 descargados).

In [12]:
import json
from pathlib import Path

FINETUNE_DATA_DIR = DATA_DIR / "finetune"
FINETUNE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Formateando train split a JSONL...")
train_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_train.jsonl"

with open(train_jsonl_path, "w", encoding="utf-8") as f:
    for i in tqdm(range(len(train_dataset)), desc="Escribiendo train JSONL"):
        sample = train_dataset[i]
        record = {
            "text": f"Speaker 1: {sample['sentence']}",
            "audio": sample["audio"]["path"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Train JSONL guardado: {train_jsonl_path}")
print(f"Tamano: {train_jsonl_path.stat().st_size / (1024**2):.1f} MB")

# --- Validation subset (2K del train para monitoreo) ---
print("\nCreando validation JSONL (muestra de 2000 registros)...")
val_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_val.jsonl"
N_VAL = min(2000, len(train_dataset))
val_indices_sample = sorted(random.sample(range(len(train_dataset)), N_VAL))

with open(val_jsonl_path, "w", encoding="utf-8") as f:
    for idx in tqdm(val_indices_sample, desc="Escribiendo val JSONL"):
        sample = train_dataset[idx]
        record = {
            "text": f"Speaker 1: {sample['sentence']}",
            "audio": sample["audio"]["path"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Validation JSONL guardado: {val_jsonl_path}")
print(f"Tamano: {val_jsonl_path.stat().st_size / (1024**2):.1f} MB")

Formateando train split a JSONL...


Escribiendo train JSONL: 100%|██████████| 336846/336846 [19:46<00:00, 283.80it/s] 


Train JSONL guardado: /home/alfrog/projects/VibeVoice_Optimization/data/finetune/cv17_es_train.jsonl
Tamano: 85.0 MB

Creando validation JSONL (muestra de 2000 registros)...


Escribiendo val JSONL: 100%|██████████| 2000/2000 [00:05<00:00, 335.11it/s]

Validation JSONL guardado: /home/alfrog/projects/VibeVoice_Optimization/data/finetune/cv17_es_val.jsonl
Tamano: 0.5 MB


---
## TSK-1.5.2: Script de Fine-Tuning para 8GB VRAM

Parametros optimizados para RTX 4060 Ti (8GB):

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `per_device_train_batch_size` | 1 | Limitar VRAM de activaciones |
| `gradient_accumulation_steps` | 64 | Batch efectivo = 64 |
| `bf16` | True | Mixed precision (Ada Lovelace) |
| `ddpm_batch_mul` | 1 | Minimo; sin amplificacion de difusion |
| `lora_r` | 8 | Rank estandar para adaptacion |
| `learning_rate` | 2.5e-5 | Recomendado por la comunidad |
| `num_train_epochs` | 1 | ~5,263 pasos (~7-15h) |
| `voice_prompt_drop_rate` | 1.0 | Drop siempre; prioriza sintesis |

El script generado se guarda en `scripts/run_finetune_es.sh`.

In [13]:
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

VENV_PYTHON = "/home/alfrog/micromamba/envs/vibevoice/bin/python"
MODEL_PATH = str(WEIGHTS_DIR / "vibevoice-1.5b")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "finetune_vibevoice_es")
TRAIN_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_train.jsonl")
VAL_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_val.jsonl")

train_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Fine-Tuning VibeVoice 1.5B -> Espanol (LoRA)
# Hardware: RTX 4060 Ti 8GB VRAM
# Duracion estimada: 7-15 horas (1 epoch, 336K muestras)
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Iniciando Fine-Tuning VibeVoice-ES ==="
echo "Modelo base : {MODEL_PATH}"
echo "Train JSONL : {TRAIN_JSONL}"
echo "Val JSONL   : {VAL_JSONL}"
echo "Output      : {OUTPUT_DIR}"
echo ""

# Limpiar cache CUDA antes de empezar
{VENV_PYTHON} -c "import torch; torch.cuda.empty_cache(); print(f'VRAM libre: {{torch.cuda.memory_allocated()/1024**3:.2f}} GB')"

{VENV_PYTHON} -m vibevoice.finetune.train_vibevoice \\
    --model_name_or_path {MODEL_PATH} \\
    --train_jsonl {TRAIN_JSONL} \\
    --validation_jsonl {VAL_JSONL} \\
    --output_dir {OUTPUT_DIR} \\
    --text_column_name text \\
    --audio_column_name audio \\
    --per_device_train_batch_size 1 \\
    --gradient_accumulation_steps 64 \\
    --learning_rate 2.5e-5 \\
    --lr_scheduler_type cosine \\
    --warmup_ratio 0.03 \\
    --num_train_epochs 1 \\
    --logging_steps 50 \\
    --save_steps 500 \\
    --eval_steps 500 \\
    --save_total_limit 3 \\
    --bf16 True \\
    --do_train \\
    --do_eval \\
    --dataloader_num_workers 2 \\
    --remove_unused_columns False \\
    --gradient_checkpointing False \\
    --ddpm_batch_mul 1 \\
    --diffusion_loss_weight 1.0 \\
    --train_diffusion_head True \\
    --ce_loss_weight 1.0  # mas peso a precision textual \\
    --voice_prompt_drop_rate 1.0 \\
    --lora_r 8 \\
    --lora_alpha 32 \\
    --lora_dropout 0.05 \\
    --lora_target_modules q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj \\
    --max_grad_norm 0.8 \\
    --gradient_clipping \\
    --seed 42

echo ""
echo "=== Fine-Tuning completado ==="
echo "Checkpoint en: {OUTPUT_DIR}"
"""

script_path = SCRIPTS_DIR / "run_finetune_es.sh"
script_path.write_text(train_script)
script_path.chmod(0o755)

print(f"Script de entrenamiento generado: {script_path}")
print(f"\nPara ejecutar:\n  bash {script_path}")

Script de entrenamiento generado: /home/alfrog/projects/VibeVoice_Optimization/scripts/run_finetune_es.sh

Para ejecutar:
  bash /home/alfrog/projects/VibeVoice_Optimization/scripts/run_finetune_es.sh


---
## TSK-1.5.3: Merge de LoRA al Modelo Base

Al finalizar el fine-tuning, los adaptadores LoRA y el diffusion head entrenado
deben fusionarse al modelo base para obtener el checkpoint **VibeVoice-ES** en
`weights/vibevoice-1.5b-es/`. Se usa el script nativo `merge_vibevoice_models.py`.

In [14]:
MERGE_OUTPUT = str(WEIGHTS_DIR / "vibevoice-1.5b-es")

merge_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Merge LoRA -> VibeVoice-ES
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Mergeando LoRA + Diffusion Head al modelo base ==="
echo "Checkpoint LoRA : {OUTPUT_DIR}"
echo "Modelo base     : {MODEL_PATH}"
echo "Output (ES)     : {MERGE_OUTPUT}"
echo ""

{VENV_PYTHON} -m vibevoice.scripts.merge_vibevoice_models \\
    --base_model_path {MODEL_PATH} \\
    --checkpoint_path {OUTPUT_DIR}/lora \\
    --output_path {MERGE_OUTPUT} \\
    --output_format safetensors

echo ""
echo "=== VibeVoice-ES generado en: {MERGE_OUTPUT} ==="
ls -lh {MERGE_OUTPUT}/*.safetensors 2>/dev/null || echo "(verificar estructura)"
"""

merge_script_path = SCRIPTS_DIR / "merge_es_checkpoint.sh"
merge_script_path.write_text(merge_script)
merge_script_path.chmod(0o755)

print(f"Script de merge generado: {merge_script_path}")
print(f"\nPara ejecutar (despues del entrenamiento):\n  bash {merge_script_path}")

Script de merge generado: /home/alfrog/projects/VibeVoice_Optimization/scripts/merge_es_checkpoint.sh

Para ejecutar (despues del entrenamiento):
  bash /home/alfrog/projects/VibeVoice_Optimization/scripts/merge_es_checkpoint.sh


---
## TSK-1.5.4: Validacion del Modelo Fine-Tuneado

Verificamos que **VibeVoice-ES** carga correctamente en VRAM y registramos
el nuevo consumo base. La evaluacion completa de calidad se realiza en Sprint 2.

In [15]:
import sys, os
repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import torch
import transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration

MODEL_ES_PATH = "../weights/vibevoice-1.5b-es"

print("Cargando VibeVoice-ES...")
torch.cuda.empty_cache()
model_es = VibeVoiceForConditionalGeneration.from_pretrained(
    MODEL_ES_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

vram_es = torch.cuda.memory_allocated() / 1024**3
print(f"VibeVoice-ES cargado. VRAM: {vram_es:.2f} GB")

# --- Verificacion de tamano en disco ---
import pathlib
model_size_mb = sum(f.stat().st_size for f in pathlib.Path(MODEL_ES_PATH).rglob("*.safetensors")) / (1024**2)
print(f"Tamano en disco (VibeVoice-ES): {model_size_mb:.1f} MB")

if vram_es < 6.0:
    print("VRAM OK - margen suficiente para cuantizacion.")
else:
    print("ADVERTENCIA: VRAM elevada. Verificar que el merge fue exitoso.")

APEX FusedRMSNorm not available, using native implementation


Cargando VibeVoice-ES...
Tied input and output embeddings using standard assignment.


Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.44s/it]

Tied input and output embeddings using standard assignment.
VibeVoice-ES cargado. VRAM: 5.04 GB
Tamano en disco (VibeVoice-ES): 10315.2 MB
VRAM OK - margen suficiente para cuantizacion.


## pre-sprint antes de los demas (2,3,4,...)

In [1]:
# ================================================================
# FASE A: Extraccion del Qwen2 (checkpoint standalone)
# Compartido por todos los Sprints 2-8
# Ejecutar UNA SOLA VEZ antes de cualquier sprint
# ================================================================

import os, sys, gc, shutil
import torch, torch.nn as nn
from pathlib import Path

for _v in ['model','qwen2_causal']:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()

repo_path = os.path.abspath("../VibeVoice_repo")
if repo_path not in sys.path: sys.path.insert(0, repo_path)

import transformers
if hasattr(transformers,"CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer",None)
    if hasattr(transformers.CONFIG_MAPPING,"_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer",None)

from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
from transformers import Qwen2ForCausalLM

MODEL_ES_PATH = "../weights/vibevoice-1.5b-es"
QWEN2_STANDALONE = Path("../weights/standalone_qwen_fp16")

print("Cargando VibeVoice-ES...")
model = VibeVoiceForConditionalGeneration.from_pretrained(
    MODEL_ES_PATH, torch_dtype=torch.float16,
    device_map={"": "cuda:0"}, attn_implementation="sdpa",
)
model.eval()
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Extraer Qwen2Model y crear Qwen2ForCausalLM
qwen2_causal = Qwen2ForCausalLM(model.model.language_model.config)
qwen2_causal.model.load_state_dict(model.model.language_model.state_dict())
# lm_head tied con embed_tokens
qwen2_causal.lm_head.weight = model.model.language_model.embed_tokens.weight
print("Qwen2ForCausalLM ensamblado")

# Guardar como checkpoint standalone
if QWEN2_STANDALONE.exists(): shutil.rmtree(QWEN2_STANDALONE)
QWEN2_STANDALONE.mkdir(parents=True, exist_ok=True)
qwen2_causal.save_pretrained(str(QWEN2_STANDALONE), safe_serialization=True)

# Guardar tokenizer (necesario para calibracion de texto)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
processor.tokenizer.save_pretrained(str(QWEN2_STANDALONE))

del model, qwen2_causal, processor
gc.collect(); torch.cuda.empty_cache()
print(f"Qwen2 standalone guardado en: {QWEN2_STANDALONE}")
for f in sorted(QWEN2_STANDALONE.glob("*.safetensors")):
    print(f"  {f.name}: {f.stat().st_size/(1024**2):.0f} MB")


/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
APEX FusedRMSNorm not available, using native implementation
/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/processor/vibevoice_asr_processor.py:23: UserWarning: audio_utils not available, will fall back to soundfile for audio loading
  warnings.warn("audio_utils not available, will fall back to soundfile for audio loading")


Cargando VibeVoice-ES...
Tied input and output embeddings using standard assignment.


Loading checkpoint shards: 100%|██████████| 3/3 [00:15<00:00,  5.01s/it]


Tied input and output embeddings using standard assignment.
VRAM: 5.04 GB
Qwen2ForCausalLM ensamblado


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Qwen2Tokenizer'. 
The class this function is called from is 'VibeVoiceTextTokenizerFast'.


Qwen2 standalone guardado en: ../weights/standalone_qwen_fp16
  model-00001-of-00002.safetensors: 4765 MB
  model-00002-of-00002.safetensors: 1569 MB


---
# Sprint 2: RTN-INT4 via bitsandbytes (DevParker baseline)

> DevParker approach: BitsAndBytesConfig(fp4) directo con llm_int8_skip_modules.
> Sin extraer Qwen2. Audio modules protegidos en FP16.
> NOTA: DevParker encontro que fp4 produce ruido estatico. Esta es la baseline.
> La solucion final de DevParker fue NF4 (Sprint 6), no fp4.


In [2]:
# === Sprint 2: RTN via bitsandbytes directo (DevParker baseline) ===

import os, sys, gc, shutil, json, time
import torch; import numpy as np
from pathlib import Path

for _v in ["model","model_rtn"]:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()

sys.path.insert(0, os.path.abspath("../VibeVoice_repo"))
import transformers
if hasattr(transformers,"CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer",None)
    if hasattr(transformers.CONFIG_MAPPING,"_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer",None)

from transformers import BitsAndBytesConfig
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
import librosa, jiwer, whisper

MODEL_ES_PATH = "../weights/vibevoice-1.5b-es"
RTN_OUTPUT = Path("../weights/vibevoice-1.5b-es-rtn")
TARGET_SR = 24000

# 1. Cuantizar directo con bnb + skip_modules
print("Cuantizando VibeVoice-ES con bnb RTN (fp4) + llm_int8_skip_modules...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="fp4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
    llm_int8_skip_modules=[
        "lm_head","prediction_head",
        "acoustic_connector","semantic_connector",
        "acoustic_tokenizer","semantic_tokenizer",
    ],
)
model_rtn = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ES_PATH, quantization_config=bnb_config,
    device_map="auto"  # dynamic offload como DevParker para OOM, attn_implementation="sdpa", torch_dtype=torch.float16,
)
model_rtn.eval(); model_rtn.set_ddpm_inference_steps(num_steps=20)
vram_rtn_idle = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM reposo RTN: {vram_rtn_idle:.2f} GB")

# 2. Guardar
if RTN_OUTPUT.exists(): shutil.rmtree(RTN_OUTPUT)
RTN_OUTPUT.mkdir(parents=True, exist_ok=True)
model_rtn.save_pretrained(str(RTN_OUTPUT), safe_serialization=True)
proc = VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
proc.save_pretrained(str(RTN_OUTPUT))
print(f"Guardado: {RTN_OUTPUT}")

# 3. Benchmark TTS
test_texts = [
    "Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.",
    "La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.",
    "El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.",
    "Los modelos de lenguaje pueden generar texto y voz con gran precision.",
]
all_wer = []; all_cer = []; all_rtf = []
wm = whisper.load_model("large-v3")
for test_text in test_texts:
    print(f"Texto: {test_text[:60]}...")
    t_start = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    inp = proc(text=[f"Speaker 1: {test_text}"], voice_samples=None, padding=True, return_tensors="pt", return_attention_mask=True)
    inp = {k: (v.to(model_rtn.device) if torch.is_tensor(v) else v) for k,v in inp.items() if v is not None}
    with torch.no_grad():
        out = model_rtn.generate(**inp, max_new_tokens=600, tokenizer=proc.tokenizer,
                                  generation_config={"do_sample": False}, verbose=False)
    t_end = time.perf_counter()
    if out.speech_outputs and out.speech_outputs[0] is not None:
        a = out.speech_outputs[0].squeeze().cpu().float().numpy()
        d = len(a) / TARGET_SR
        all_rtf.append((t_end-t_start)/d)
        a16 = librosa.resample(a.astype(np.float32), orig_sr=TARGET_SR, target_sr=16000)
        trans = wm.transcribe(a16, language="es", fp16=True)["text"].strip()
        all_wer.append(jiwer.wer(test_text.lower(), trans.lower()))
        all_cer.append(jiwer.cer(test_text.lower(), trans.lower()))
        print(f"  Audio: {d:.1f}s | RTF: {all_rtf[-1]:.2f} | WER={all_wer[-1]:.3f} | {trans[:60]}")
    else:
        all_wer.append(1.0); all_cer.append(1.0); all_rtf.append(float('inf'))
        print("  ERROR: sin audio")
vram_peak_rtn = torch.cuda.max_memory_allocated() / 1024**3
rtf_rtn = np.mean(all_rtf) if all_rtf else float('inf')
wer_rtn = np.mean(all_wer) if all_wer else 1.0
cer_rtn = np.mean(all_cer) if all_cer else 1.0
del wm, out; gc.collect(); torch.cuda.empty_cache()
print(f"\n=== RTN === VRAM peak: {vram_peak_rtn:.2f} GB | RTF: {rtf_rtn:.4f} | WER: {wer_rtn:.4f} | CER: {cer_rtn:.4f}")


Cuantizando VibeVoice-ES con bnb RTN (fp4) + llm_int8_skip_modules...


Loading checkpoint shards: 100%|██████████| 3/3 [00:14<00:00,  4.69s/it]


VRAM reposo RTN: 3.31 GB


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Qwen2Tokenizer'. 
The class this function is called from is 'VibeVoiceTextTokenizerFast'.


Guardado: ../weights/vibevoice-1.5b-es-rtn
Texto: Hola, buenos dias. Este es un modelo de sintesis de voz en e...


Generating (active: 1/1):   0%|          | 0/102 [00:00<?, ?it/s]/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  Audio: 6.3s | RTF: 1.19 | WER=0.231 | Hola, buenos días. Este es un modelo de síntesis de voz en e
Texto: La inteligencia artificial permite crear sistemas de voz cad...


  Audio: 7.3s | RTF: 1.10 | WER=0.333 | La inteligencia permite crear falsos temas de voz cada vez m
Texto: El aprendizaje profundo ha revolucionado el procesamiento de...


  Audio: 5.2s | RTF: 1.12 | WER=0.333 | El aprendizaje de los softwandos ha revolucionado el procesa
Texto: Los modelos de lenguaje pueden generar texto y voz con gran ...


  Audio: 5.5s | RTF: 1.12 | WER=0.750 | Recuerden que el lenguaje puede generar textos y buen foco e

=== RTN === VRAM peak: 9.55 GB | RTF: 1.1308 | WER: 0.4119 | CER: 0.1994


---
# Sprint 3: GPTQ via GPTQModel (qforge)

> qforge: transformers>=5.4.0 con GPTQModel.
> GPTQModel en qforge: INT4 g128 Hessiana con damp_percent=0.1.
> vibevoice: reemplazo language_model via _sfload + benchmark.


In [ ]:
# === Sprint 3: GPTQ via GPTQModel (qforge) + benchmark vibevoice ===

# Estrategia: GPTQModel cuantiza Qwen2 en qforge
# vibevoice carga decoder desde safetensors, reemplaza language_model
# Resultado esperado: disco -83%, WER ~0.14, VRAM 5.04

import os, sys, gc, shutil, subprocess, json, time
import torch; import numpy as np
from pathlib import Path

for _v in ["model","model_gptq"]:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()

QWEN2_STANDALONE = Path("../weights/standalone_qwen_fp16")
GPTQ_OUTPUT = Path("../weights/vibevoice-1.5b-es-gptq")
MODEL_ES_PATH = "../weights/vibevoice-1.5b-es"

# Paso A: GPTQModel en qforge
print("=== Paso A: GPTQModel en qforge ===\n")
script = f'''
import os, torch, gc, shutil; gc.collect(); torch.cuda.empty_cache()
from pathlib import Path
from transformers import AutoTokenizer
from datasets import load_dataset, Dataset
from gptqmodel import GPTQModel, QuantizeConfig
import glob as _glob

MODEL_PATH = '{QWEN2_STANDALONE}'
OUTPUT_DIR = '{GPTQ_OUTPUT / "decoder-gptq"}'

cv17 = load_dataset('fsicoli/common_voice_17_0','es',split='train')
cache_dir = os.path.expanduser('~/.cache/huggingface/datasets/fsicoli___common_voice_17_0/es/16.1.0')
arrow_files = _glob.glob(cache_dir + '/**/*.arrow', recursive=True)
ds = Dataset.from_file(sorted(arrow_files)[0])
sentences = [s for s in ds['sentence'] if isinstance(s,str) and len(s)>20][:3000]
chunk, chunk_len, calib = [], 0, []
for s in sentences:
    chunk.append(s); chunk_len += len(s.split())
    if chunk_len >= 200: calib.append(' '.join(chunk)); chunk, chunk_len = [], 0
if chunk: calib.append(' '.join(chunk))
print(f'Calib: {{len(calib)}} chunks')

model = GPTQModel.from_pretrained(MODEL_PATH, quantize_config=QuantizeConfig(bits=4, group_size=128, sym=True, damp_percent=0.1, desc_act=False))
model.quantize(calib[:256], batch_size=1)
if Path(OUTPUT_DIR).exists(): shutil.rmtree(OUTPUT_DIR)
Path(OUTPUT_DIR).mkdir(parents=True)
model.save_quantized(OUTPUT_DIR)
tok = AutoTokenizer.from_pretrained(MODEL_PATH); tok.save_pretrained(OUTPUT_DIR)
print(f'GPTQ done: {{OUTPUT_DIR}}')
'''
with open("/tmp/_gptq_s3.py","w") as f: f.write(script)
result = subprocess.run(["/home/alfrog/.local/bin/micromamba","run","-n","qforge","python","/tmp/_gptq_s3.py"])
os.remove("/tmp/_gptq_s3.py")
gptq_done = result.returncode == 0
print(f"GPTQModel exit: {result.returncode}")

if not gptq_done:
    print('ERROR')
else:
    print("\n=== Paso B: Cargar decoder GPTQ + VibeVoice ===\n")
    sys.path.insert(0, os.path.abspath("../VibeVoice_repo"))
    import transformers
    if hasattr(transformers,"CONFIG_MAPPING"):
        transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer",None)
        if hasattr(transformers.CONFIG_MAPPING,"_mapping"): transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer",None)
    from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
    from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
    from safetensors.torch import load_file as _sfload
    import librosa, jiwer, whisper

    model_inf = VibeVoiceForConditionalGenerationInference.from_pretrained(MODEL_ES_PATH, torch_dtype=torch.float16, device_map={"": "cuda:0"}, attn_implementation="sdpa")
    model_inf.eval(); model_inf.set_ddpm_inference_steps(num_steps=20)
    gptq_sd = _sfload(str(GPTQ_OUTPUT / "decoder-gptq" / "model.safetensors"))
    lm_sd = model_inf.model.language_model.state_dict()
    for k in list(lm_sd.keys()):
        gptq_key = "model." + k if not k.startswith("model.") else k
        if gptq_key in gptq_sd: lm_sd[k] = gptq_sd[gptq_key].to(torch.float16).to(model_inf.device)
    model_inf.model.language_model.load_state_dict(lm_sd, strict=False)
    del gptq_sd, lm_sd; gc.collect(); torch.cuda.empty_cache()
    model_inf.save_pretrained(str(GPTQ_OUTPUT), safe_serialization=True)
    proc_tmp = VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
    proc_tmp.save_pretrained(str(GPTQ_OUTPUT))
    del proc_tmp
    print("Modelo unificado guardado en: GPTQ_OUTPUT")
    vram_idle = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM reposo GPTQ: {vram_idle:.2f} GB")

    TARGET_SR=24000
    test_texts=["Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.","La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.","El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.","Los modelos de lenguaje pueden generar texto y voz con gran precision."]
    processor_inf=VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
    all_wer=[]; all_cer=[]; all_rtf=[]; wm=whisper.load_model("large-v3")
    for test_text in test_texts:
        t_start=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
        inp=processor_inf(text=[f"Speaker 1: {test_text}"],voice_samples=None,padding=True,return_tensors="pt",return_attention_mask=True)
        inp={k:(v.to(model_inf.device) if torch.is_tensor(v) else v) for k,v in inp.items() if v is not None}
        with torch.no_grad(): out=model_inf.generate(**inp,max_new_tokens=600,tokenizer=processor_inf.tokenizer,generation_config={"do_sample":False},verbose=False)
        t_end=time.perf_counter()
        if out.speech_outputs and out.speech_outputs[0] is not None:
            a=out.speech_outputs[0].squeeze().cpu().float().numpy(); d=len(a)/TARGET_SR; all_rtf.append((t_end-t_start)/d)
            a16=librosa.resample(a.astype(np.float32),orig_sr=TARGET_SR,target_sr=16000)
            trans=wm.transcribe(a16,language="es",fp16=True)["text"].strip()
            all_wer.append(jiwer.wer(test_text.lower(),trans.lower())); all_cer.append(jiwer.cer(test_text.lower(),trans.lower()))
            print(f"  Audio: {d:.1f}s | RTF: {all_rtf[-1]:.2f} | WER={all_wer[-1]:.3f} | {trans[:60]}")
        else:
            all_wer.append(1.0); all_cer.append(1.0); all_rtf.append(float('inf')); print('  ERROR')
    vram_peak=torch.cuda.max_memory_allocated()/1024**3; rtf=np.mean(all_rtf) if all_rtf else float('inf')
    wer_gptq=np.mean(all_wer) if all_wer else 1.0; cer_gptq=np.mean(all_cer) if all_cer else 1.0
    del wm,out,model_inf; gc.collect(); torch.cuda.empty_cache()
    print(f"\n=== GPTQ === VRAM peak: {vram_peak:.2f} GB | RTF: {rtf:.4f} | WER: {wer_gptq:.4f} | CER: {cer_gptq:.4f}")


---
# Sprint 4: AWQ via AutoAWQ (ncoder-ai, todo en vibevoice)

> ncoder-ai: AutoAWQ zero_point=True, g128, w4, GEMM.
> AutoAWQ instalado en vibevoice (wheel pre-compilado). Sin qforge.
> Paso A: AutoAWQ.quantize() Qwen2  |  Paso B: inyeccion + benchmark.


In [3]:
# === Sprint 4: AWQ via AutoAWQ (ncoder-ai, todo en vibevoice) ===

# ncoder-ai: AutoAWQ con zero_point=True, g128, w4, GEMM, modules_to_not_convert
# AutoAWQ instalado en vibevoice (wheel pre-compilado, sin qforge)

import os, sys, gc, shutil, json, time
import torch; import numpy as np
from pathlib import Path

for _v in ["model","model_awq","qwen2","decoder_awq"]:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()

QWEN2_STANDALONE = Path("../weights/standalone_qwen_fp16")
AWQ_OUTPUT = Path("../weights/vibevoice-1.5b-es-awq")
MODEL_ES_PATH = "../weights/vibevoice-1.5b-es"

# Paso A: Cuantizar Qwen2 standalone con AutoAWQ
_awq_ckpt = AWQ_OUTPUT / "decoder-awq"
if _awq_ckpt.exists() and any(_awq_ckpt.iterdir()):
    print(f"Paso A ya completado: {_awq_ckpt}")
else:
    print("=== Paso A: AutoAWQ quantize Qwen2 ===\n")
    from datasets import load_dataset, Dataset
    from transformers import AutoTokenizer
    from awq import AutoAWQForCausalLM
    import glob as _glob

    cv17 = load_dataset("fsicoli/common_voice_17_0","es",split="train")
    cache_dir = os.path.expanduser("~/.cache/huggingface/datasets/fsicoli___common_voice_17_0/es/16.1.0")
    arrow_files = _glob.glob(cache_dir + "/**/*.arrow", recursive=True)
    ds = Dataset.from_file(sorted(arrow_files)[0])
    sentences = [s for s in ds["sentence"] if isinstance(s,str) and len(s)>20][:3000]
    chunk, chunk_len, calib = [], 0, []
    for s in sentences:
        chunk.append(s); chunk_len += len(s.split())
        if chunk_len >= 200: calib.append(" ".join(chunk)); chunk, chunk_len = [], 0
    if chunk: calib.append(" ".join(chunk))
    print(f"Calib: {len(calib)} chunks")

    model = AutoAWQForCausalLM.from_pretrained(str(QWEN2_STANDALONE), safetensors=True)
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)  # base tokenizer
    quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
    model.quantize(tokenizer, quant_config=quant_config)

    (AWQ_OUTPUT / "decoder-awq").mkdir(parents=True, exist_ok=True)
    model.save_quantized(str(AWQ_OUTPUT / "decoder-awq"))
    tokenizer.save_pretrained(str(AWQ_OUTPUT / "decoder-awq"))
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"AWQ done: {AWQ_OUTPUT / 'decoder-awq'}")

# Paso B: Inyeccion + benchmark (mismo patron que Sprint 3)
print("\n=== Paso B: Cargar decoder AWQ + VibeVoice + benchmark ===\n")
sys.path.insert(0, os.path.abspath("../VibeVoice_repo"))
from awq import AutoAWQForCausalLM
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
import librosa, jiwer, whisper
print("Cargando decoder AWQ con AutoAWQ nativo...")
decoder_awq = AutoAWQForCausalLM.from_quantized(
    str(AWQ_OUTPUT / "decoder-awq"),
    device_map={"": "cuda:0"},
)
vram_dec = torch.cuda.memory_allocated() / 1024**3
print(f"Decoder AWQ (INT4): {vram_dec:.2f} GB")

print("Cargando VibeVoice-ES...")
model_inf = VibeVoiceForConditionalGenerationInference.from_pretrained(MODEL_ES_PATH, torch_dtype=torch.float16, device_map={"": "cuda:0"}, attn_implementation="sdpa")
model_inf.eval(); model_inf.set_ddpm_inference_steps(num_steps=20)
# Inyeccion: language_model = decoder AWQ (WQLinear layers)
model_inf.model.language_model = decoder_awq.model.model
model_inf.lm_head.weight = decoder_awq.model.model.embed_tokens.weight
del decoder_awq; gc.collect(); torch.cuda.empty_cache()
vram_idle = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM reposo AWQ: {vram_idle:.2f} GB")

# Guardar modelo portable
model_inf.save_pretrained(str(AWQ_OUTPUT), safe_serialization=True)
proc_tmp = VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
proc_tmp.save_pretrained(str(AWQ_OUTPUT))
del proc_tmp
print(f"Modelo portable guardado: {AWQ_OUTPUT}")

# Benchmark TTS
TARGET_SR=24000
test_texts=["Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.","La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.","El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.","Los modelos de lenguaje pueden generar texto y voz con gran precision."]
processor_inf=VibeVoiceProcessor.from_pretrained(MODEL_ES_PATH)
all_wer=[]; all_cer=[]; all_rtf=[]; wm=whisper.load_model("large-v3")
for test_text in test_texts:
    t_start=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    inp=processor_inf(text=[f"Speaker 1: {test_text}"],voice_samples=None,padding=True,return_tensors="pt",return_attention_mask=True)
    inp={k:(v.to(model_inf.device) if torch.is_tensor(v) else v) for k,v in inp.items() if v is not None}
    with torch.no_grad(): out=model_inf.generate(**inp,max_new_tokens=600,tokenizer=processor_inf.tokenizer,generation_config={"do_sample":False},verbose=False)
    t_end=time.perf_counter()
    if out.speech_outputs and out.speech_outputs[0] is not None:
        a=out.speech_outputs[0].squeeze().cpu().float().numpy(); d=len(a)/TARGET_SR; all_rtf.append((t_end-t_start)/d)
        a16=librosa.resample(a.astype(np.float32),orig_sr=TARGET_SR,target_sr=16000)
        trans=wm.transcribe(a16,language="es",fp16=True)["text"].strip()
        all_wer.append(jiwer.wer(test_text.lower(),trans.lower())); all_cer.append(jiwer.cer(test_text.lower(),trans.lower()))
        print(f"  Audio: {d:.1f}s | RTF: {all_rtf[-1]:.2f} | WER={all_wer[-1]:.3f} | {trans[:60]}")
    else:
        all_wer.append(1.0); all_cer.append(1.0); all_rtf.append(float("inf")); print("  ERROR")
vram_peak=torch.cuda.max_memory_allocated()/1024**3; rtf=np.mean(all_rtf) if all_rtf else float("inf")
wer_awq=np.mean(all_wer) if all_wer else 1.0; cer_awq=np.mean(all_cer) if all_cer else 1.0
del wm,out,model_inf; gc.collect(); torch.cuda.empty_cache()
print(f"\n=== AWQ === VRAM peak: {vram_peak:.2f} GB | RTF: {rtf:.4f} | WER: {wer_awq:.4f} | CER: {cer_awq:.4f}")



Paso A ya completado: ../weights/vibevoice-1.5b-es-awq/decoder-awq

=== Paso B: Cargar decoder AWQ + VibeVoice + benchmark ===



/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/processor/vibevoice_asr_processor.py:23: UserWarning: audio_utils not available, will fall back to soundfile for audio loading
  warnings.warn("audio_utils not available, will fall back to soundfile for audio loading")


Cargando decoder AWQ con AutoAWQ nativo...


Replacing layers...: 100%|██████████| 28/28 [00:03<00:00,  7.69it/s]
/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/awq/models/base.py:541: UserWarning: Skipping fusing modules because AWQ extension is not installed.No module named 'awq_ext'
  warnings.warn("Skipping fusing modules because AWQ extension is not installed." + msg)


Decoder AWQ (INT4): 1.09 GB
Cargando VibeVoice-ES...


...

  Audio: 7.9s | RTF: 1.90 | WER=1.154 | Hola, buenos días. Es el pueblo muy unido entre los formador


Generating (active: 1/1):   0%|          | 0/92 [00:00<?, ?it/s]/home/alfrog/projects/VibeVoice_Optimization/VibeVoice_repo/vibevoice/schedule/dpm_solver.py:384: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  sigmas = np.array(((1 - self.alphas_cumprod) / self.alphas_cumprod) ** 0.5)


  Audio: 6.8s | RTF: 1.40 | WER=0.333 | La inteligencia artificial también crea sistemas de box cada


  Audio: 7.6s | RTF: 1.45 | WER=1.000 | Ahora el mensaje profundo ha reconocido la lengua de la dict


  Audio: 7.9s | RTF: 1.33 | WER=0.167 | Los modelos de lenguaje pueden generar texto y voz a gran pr

=== AWQ === VRAM peak: 9.48 GB | RTF: 1.5228 | WER: 0.6635 | CER: 0.3978


---
# Sprint 5: INT8 via bitsandbytes

> vibevoice: BitsAndBytesConfig(load_in_8bit=True).
> Cuantiza Qwen2 standalone. Split layout: decoder-int8/.
> Preserva prosodia con mapeo conservador de canales.


In [ ]:
# === Sprint 5: INT8 via bitsandbytes ===
import os, sys, gc, shutil, json, time
import torch; import numpy as np
from pathlib import Path
for _v in ['model']:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
QWEN2 = Path("../weights/standalone_qwen_fp16"); INT8_OUT = Path("../weights/vibevoice-1.5b-es-int8"); MODEL_ES = "../weights/vibevoice-1.5b-es"
print("Cuantizando Qwen2 con bnb INT8...")
q8 = AutoModelForCausalLM.from_pretrained(str(QWEN2), quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map={"": "cuda:0"})
q8.eval(); vram_bnb = torch.cuda.memory_allocated()/1024**3
print(f"VRAM Qwen2 INT8: {vram_bnb:.2f} GB")
int8_sd = {}
for n, p in q8.named_parameters():
    if not p.requires_grad: int8_sd[n] = p.data.to(torch.float16).cpu().clone()
del q8; gc.collect(); torch.cuda.empty_cache()
if INT8_OUT.exists(): shutil.rmtree(INT8_OUT)
INT8_OUT.mkdir(parents=True, exist_ok=True); (INT8_OUT/"decoder-int8").mkdir()
for f in QWEN2.glob("*.json"): shutil.copy2(str(f), str(INT8_OUT/"decoder-int8"/f.name))
for f in QWEN2.glob("tokenizer*"):
    if f.is_file(): shutil.copy2(str(f), str(INT8_OUT/"decoder-int8"/f.name))
torch.save(int8_sd, INT8_OUT/"decoder-int8"/"model.pt")
sys.path.insert(0, os.path.abspath("../VibeVoice_repo"))
from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration
m = VibeVoiceForConditionalGeneration.from_pretrained(MODEL_ES, torch_dtype=torch.float16, device_map={"":"cuda:0"}, attn_implementation="sdpa")
m.eval(); nd_sd = {}
for k, v in m.state_dict().items():
    if not k.startswith("model.language_model.") or k.startswith("model.language_model.embed_tokens."): nd_sd[k] = v.cpu()
from safetensors.torch import save_file
save_file({k:v for k,v in nd_sd.items()}, INT8_OUT / "model.safetensors")
import json as _json
cfg = _json.load(open(Path(MODEL_ES)/"config.json")); cfg["vibevoice_decoder_model_path"]="decoder-int8"; cfg["vibevoice_decoder_quantization"]="int8"
with open(INT8_OUT/"config.json","w") as f: _json.dump(cfg,f,indent=2)
for f in Path(MODEL_ES).glob("preprocessor*"): shutil.copy2(str(f), str(INT8_OUT/f.name))
gc.collect(); torch.cuda.empty_cache()
# Benchmark
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
import librosa, jiwer, whisper
m_int8 = VibeVoiceForConditionalGenerationInference.from_pretrained(str(INT8_OUT), torch_dtype=torch.float16, device_map={"":"cuda:0"}, attn_implementation="sdpa")
m_int8.eval(); m_int8.set_ddpm_inference_steps(num_steps=20)
vram_int8_idle = torch.cuda.memory_allocated()/1024**3
    # Guardar modelo portable
    if INT8_OUT.exists(): shutil.rmtree(INT8_OUT)
    INT8_OUT.mkdir(parents=True, exist_ok=True)
    print("Guardando modelo portable...")
print(f"VRAM reposo INT8: {vram_int8_idle:.2f} GB")
TARGET_SR=24000; p_inf=VibeVoiceProcessor.from_pretrained(MODEL_ES)
tt=["Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.","La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.","El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.","Los modelos de lenguaje pueden generar texto y voz con gran precision."]
au=[]; rt=[]
for tx in tt:
    print(f"Texto: {tx[:60]}...")
    ts=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    ip=p_inf(text=[f"Speaker 1: {tx}"],voice_samples=None,padding=True,return_tensors="pt",return_attention_mask=True)
    ip={k:(v.to(m_int8.device) if torch.is_tensor(v) else v) for k,v in ip.items() if v is not None}
    with torch.no_grad(): o=m_int8.generate(**ip,max_new_tokens=600,tokenizer=p_inf.tokenizer,generation_config={"do_sample":False},verbose=False)
    te=time.perf_counter()
    if o.speech_outputs and o.speech_outputs[0] is not None:
        a=o.speech_outputs[0].squeeze().cpu().float().numpy(); d=len(a)/TARGET_SR
        rt.append((te-ts)/d); au.append(a); print(f"  Audio: {d:.1f}s | RTF: {(te-ts)/d:.2f}")
    else: print("  ERROR"); au.append(None)
vram_peak_int8=torch.cuda.max_memory_allocated()/1024**3; rtf_int8=np.mean(rt) if rt else float('inf')
del m_int8; gc.collect(); torch.cuda.empty_cache()
print(f"VRAM peak INT8: {vram_peak_int8:.2f} GB | RTF: {rtf_int8:.4f}")
wm=whisper.load_model("large-v3"); aw=[]; ac=[]
for i,(tx,a) in enumerate(zip(tt,au)):
    if a is None: aw.append(1.0); ac.append(1.0); continue
    a16=librosa.resample(a.astype(np.float32),orig_sr=TARGET_SR,target_sr=16000)
    tr=wm.transcribe(a16,language="es",fp16=True)["text"].strip()
    aw.append(jiwer.wer(tx.lower(),tr.lower())); ac.append(jiwer.cer(tx.lower(),tr.lower()))
    print(f"[{i+1}] WER={aw[-1]:.3f} CER={ac[-1]:.3f} | {tr[:80]}")
    m_int8.save_pretrained(str(INT8_OUT), safe_serialization=True)  # portable
    print('Modelo portable guardado: INT8_OUT')
del wm; gc.collect(); torch.cuda.empty_cache()
wer_int8=np.mean(aw) if aw else 1.0; cer_int8=np.mean(ac) if ac else 1.0
print(f"WER INT8: {wer_int8:.4f} | CER INT8: {cer_int8:.4f}")


---
# Sprint 6: NF4 + Double Quant via bitsandbytes

> vibevoice: BitsAndBytesConfig(nf4, double_quant, bfloat16 compute).
> Excluye embed_tokens. Preserva diarizacion de hablantes.


In [ ]:
# === Sprint 6: NF4 + Double Quant ===
import os, sys, gc, shutil, json, time
import torch; import numpy as np
from pathlib import Path
for _v in ['model']:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
QWEN2 = Path("../weights/standalone_qwen_fp16"); NF4_OUT = Path("../weights/vibevoice-1.5b-es-nf4"); MODEL_ES = "../weights/vibevoice-1.5b-es"
print("Cuantizando Qwen2 con NF4+Double Quant...")
qnf4 = AutoModelForCausalLM.from_pretrained(str(QWEN2), quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16), device_map={"": "cuda:0"})
qnf4.eval(); vram_bnb = torch.cuda.memory_allocated()/1024**3
print(f"VRAM Qwen2 NF4: {vram_bnb:.2f} GB")
# Excluir embed_tokens del state_dict
nf4_sd = {}
for n, p in qnf4.named_parameters():
    if 'embed_tokens' in n: continue
    if not p.requires_grad: nf4_sd[n] = p.data.to(torch.float16).cpu().clone()
del qnf4; gc.collect(); torch.cuda.empty_cache()
if NF4_OUT.exists(): shutil.rmtree(NF4_OUT)
NF4_OUT.mkdir(parents=True, exist_ok=True); (NF4_OUT/"decoder-nf4").mkdir()
for f in QWEN2.glob("*.json"): shutil.copy2(str(f), str(NF4_OUT/"decoder-nf4"/f.name))
for f in QWEN2.glob("tokenizer*"):
    if f.is_file(): shutil.copy2(str(f), str(NF4_OUT/"decoder-nf4"/f.name))
torch.save(nf4_sd, NF4_OUT/"decoder-nf4"/"model.pt")
sys.path.insert(0, os.path.abspath("../VibeVoice_repo"))
from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration
m = VibeVoiceForConditionalGeneration.from_pretrained(MODEL_ES, torch_dtype=torch.float16, device_map={"":"cuda:0"}, attn_implementation="sdpa")
m.eval(); nd_sd = {}
for k, v in m.state_dict().items():
    if not k.startswith("model.language_model.") or k.startswith("model.language_model.embed_tokens."): nd_sd[k] = v.cpu()
from safetensors.torch import save_file
save_file({k:v for k,v in nd_sd.items()}, NF4_OUT / "model.safetensors")
import json as _json
cfg = _json.load(open(Path(MODEL_ES)/"config.json")); cfg["vibevoice_decoder_model_path"]="decoder-nf4"; cfg["vibevoice_decoder_quantization"]="nf4"
with open(NF4_OUT/"config.json","w") as f: _json.dump(cfg,f,indent=2)
for f in Path(MODEL_ES).glob("preprocessor*"): shutil.copy2(str(f), str(NF4_OUT/f.name))
gc.collect(); torch.cuda.empty_cache()
# Benchmark
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
import librosa, jiwer, whisper
m_nf4 = VibeVoiceForConditionalGenerationInference.from_pretrained(str(NF4_OUT), torch_dtype=torch.float16, device_map={"":"cuda:0"}, attn_implementation="sdpa")
m_nf4.eval(); m_nf4.set_ddpm_inference_steps(num_steps=20)
vram_nf4_idle = torch.cuda.memory_allocated()/1024**3
    # Guardar modelo portable
    if NF4_OUT.exists(): shutil.rmtree(NF4_OUT)
    NF4_OUT.mkdir(parents=True, exist_ok=True)
    print("Guardando modelo portable...")
print(f"VRAM reposo NF4: {vram_nf4_idle:.2f} GB")
TARGET_SR=24000; p_inf=VibeVoiceProcessor.from_pretrained(MODEL_ES)
tt=["Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.","La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.","El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.","Los modelos de lenguaje pueden generar texto y voz con gran precision."]
au=[]; rt=[]
for tx in tt:
    print(f"Texto: {tx[:60]}...")
    ts=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    ip=p_inf(text=[f"Speaker 1: {tx}"],voice_samples=None,padding=True,return_tensors="pt",return_attention_mask=True)
    ip={k:(v.to(m_nf4.device) if torch.is_tensor(v) else v) for k,v in ip.items() if v is not None}
    with torch.no_grad(): o=m_nf4.generate(**ip,max_new_tokens=600,tokenizer=p_inf.tokenizer,generation_config={"do_sample":False},verbose=False)
    te=time.perf_counter()
    if o.speech_outputs and o.speech_outputs[0] is not None:
        a=o.speech_outputs[0].squeeze().cpu().float().numpy(); d=len(a)/TARGET_SR
        rt.append((te-ts)/d); au.append(a); print(f"  Audio: {d:.1f}s | RTF: {(te-ts)/d:.2f}")
    else: print("  ERROR"); au.append(None)
vram_peak_nf4=torch.cuda.max_memory_allocated()/1024**3; rtf_nf4=np.mean(rt) if rt else float('inf')
del m_nf4; gc.collect(); torch.cuda.empty_cache()
print(f"VRAM peak NF4: {vram_peak_nf4:.2f} GB | RTF: {rtf_nf4:.4f}")
wm=whisper.load_model("large-v3"); aw=[]; ac=[]
for i,(tx,a) in enumerate(zip(tt,au)):
    if a is None: aw.append(1.0); ac.append(1.0); continue
    a16=librosa.resample(a.astype(np.float32),orig_sr=TARGET_SR,target_sr=16000)
    tr=wm.transcribe(a16,language="es",fp16=True)["text"].strip()
    aw.append(jiwer.wer(tx.lower(),tr.lower())); ac.append(jiwer.cer(tx.lower(),tr.lower()))
    print(f"[{i+1}] WER={aw[-1]:.3f} CER={ac[-1]:.3f} | {tr[:80]}")
    m_nf4.save_pretrained(str(NF4_OUT), safe_serialization=True)  # portable
    print('Modelo portable guardado: NF4_OUT')
del wm; gc.collect(); torch.cuda.empty_cache()
wer_nf4=np.mean(aw) if aw else 1.0; cer_nf4=np.mean(ac) if ac else 1.0
print(f"WER NF4: {wer_nf4:.4f} | CER NF4: {cer_nf4:.4f}")


---
# Sprint 7: FP8 E4M3FN via torch nativo

> vibevoice: torch.float8_e4m3fn con proteccion de rango dinamico.
> Tensor Cores Ada Lovelace 4ta Gen. Sin calibracion.
> Elimina voz metalica del DPM-Solver.


In [ ]:
# === Sprint 7: FP8 E4M3FN Nativo ===
import os, sys, gc, shutil, json, time
import torch; import numpy as np
from pathlib import Path
for _v in ['model']:
    if _v in dir():
        try: del globals()[_v]
        except: pass
gc.collect(); torch.cuda.empty_cache()
from transformers import Qwen2ForCausalLM
QWEN2 = Path("../weights/standalone_qwen_fp16"); FP8_OUT = Path("../weights/vibevoice-1.5b-es-fp8"); MODEL_ES = "../weights/vibevoice-1.5b-es"
print("Cuantizando Qwen2 a FP8 E4M3FN...")
qwen2_fp16 = Qwen2ForCausalLM.from_pretrained(str(QWEN2), torch_dtype=torch.float16)
qwen2_fp16.eval()
fp8_scales = {}  # metadata de scaling
for name, module in qwen2_fp16.named_modules():
    if isinstance(module, torch.nn.Linear):
        W = module.weight.data.float()
        mx = W.abs().max()
        if mx > 448:
            scale = mx / 448.0
            fp8_scales[name] = scale.item()
            W = W / scale
        else:
            fp8_scales[name] = 1.0
        module.weight.data = W.to(torch.float8_e4m3fn)
print(f"FP8 scales guardadas: {len(fp8_scales)}")
# Guardar Qwen2 FP8 + escalas
if FP8_OUT.exists(): shutil.rmtree(FP8_OUT)
FP8_OUT.mkdir(parents=True, exist_ok=True); (FP8_OUT/"decoder-fp8").mkdir(parents=True, exist_ok=True)
torch.save(qwen2_fp16.state_dict(), FP8_OUT/"decoder-fp8"/"model.pt")
with open(FP8_OUT/"decoder-fp8"/"fp8_scales.json", "w") as f: json.dump(fp8_scales, f, indent=2)
for f in QWEN2.glob("*.json"): shutil.copy2(str(f), str(FP8_OUT/"decoder-fp8"/f.name))
for f in QWEN2.glob("tokenizer*"):
    if f.is_file(): shutil.copy2(str(f), str(FP8_OUT/"decoder-fp8"/f.name))
del qwen2_fp16; gc.collect(); torch.cuda.empty_cache()
# Split layout + benchmark
sys.path.insert(0, os.path.abspath("../VibeVoice_repo"))
from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration
m = VibeVoiceForConditionalGeneration.from_pretrained(MODEL_ES, torch_dtype=torch.float16, device_map={"":"cuda:0"}, attn_implementation="sdpa")
m.eval(); nd_sd = {}
for k, v in m.state_dict().items():
    if not k.startswith("model.language_model.") or k.startswith("model.language_model.embed_tokens."): nd_sd[k] = v.cpu()
from safetensors.torch import save_file
save_file({k:v for k,v in nd_sd.items()}, FP8_OUT / "model.safetensors")
import json as _json
cfg = _json.load(open(Path(MODEL_ES)/"config.json")); cfg["vibevoice_decoder_model_path"]="decoder-fp8"; cfg["vibevoice_decoder_quantization"]="fp8"
with open(FP8_OUT/"config.json","w") as f: _json.dump(cfg,f,indent=2)
for f in Path(MODEL_ES).glob("preprocessor*"): shutil.copy2(str(f), str(FP8_OUT/f.name))
gc.collect(); torch.cuda.empty_cache()
# Benchmark
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
import librosa, jiwer, whisper
m_fp8 = VibeVoiceForConditionalGenerationInference.from_pretrained(str(FP8_OUT), torch_dtype=torch.float16, device_map={"":"cuda:0"}, attn_implementation="sdpa")
m_fp8.eval(); m_fp8.set_ddpm_inference_steps(num_steps=20)
vram_fp8_idle = torch.cuda.memory_allocated()/1024**3
    # Guardar modelo portable
    if FP8_OUT.exists(): shutil.rmtree(FP8_OUT)
    FP8_OUT.mkdir(parents=True, exist_ok=True)
    print("Guardando modelo portable...")
print(f"VRAM reposo FP8: {vram_fp8_idle:.2f} GB")
TARGET_SR=24000; p_inf=VibeVoiceProcessor.from_pretrained(MODEL_ES)
tt=["Hola, buenos dias. Este es un modelo de sintesis de voz en espanol.","La inteligencia artificial permite crear sistemas de voz cada vez mas naturales.","El aprendizaje profundo ha revolucionado el procesamiento del lenguaje.","Los modelos de lenguaje pueden generar texto y voz con gran precision."]
au=[]; rt=[]
for tx in tt:
    print(f"Texto: {tx[:60]}...")
    ts=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    ip=p_inf(text=[f"Speaker 1: {tx}"],voice_samples=None,padding=True,return_tensors="pt",return_attention_mask=True)
    ip={k:(v.to(m_fp8.device) if torch.is_tensor(v) else v) for k,v in ip.items() if v is not None}
    with torch.no_grad(): o=m_fp8.generate(**ip,max_new_tokens=600,tokenizer=p_inf.tokenizer,generation_config={"do_sample":False},verbose=False)
    te=time.perf_counter()
    if o.speech_outputs and o.speech_outputs[0] is not None:
        a=o.speech_outputs[0].squeeze().cpu().float().numpy(); d=len(a)/TARGET_SR
        rt.append((te-ts)/d); au.append(a); print(f"  Audio: {d:.1f}s | RTF: {(te-ts)/d:.2f}")
    else: print("  ERROR"); au.append(None)
vram_peak_fp8=torch.cuda.max_memory_allocated()/1024**3; rtf_fp8=np.mean(rt) if rt else float('inf')
del m_fp8; gc.collect(); torch.cuda.empty_cache()
print(f"VRAM peak FP8: {vram_peak_fp8:.2f} GB | RTF: {rtf_fp8:.4f}")
wm=whisper.load_model("large-v3"); aw=[]; ac=[]
for i,(tx,a) in enumerate(zip(tt,au)):
    if a is None: aw.append(1.0); ac.append(1.0); continue
    a16=librosa.resample(a.astype(np.float32),orig_sr=TARGET_SR,target_sr=16000)
    tr=wm.transcribe(a16,language="es",fp16=True)["text"].strip()
    aw.append(jiwer.wer(tx.lower(),tr.lower())); ac.append(jiwer.cer(tx.lower(),tr.lower()))
    print(f"[{i+1}] WER={aw[-1]:.3f} CER={ac[-1]:.3f} | {tr[:80]}")
    m_fp8.save_pretrained(str(FP8_OUT), safe_serialization=True)  # portable
    print('Modelo portable guardado: FP8_OUT')
del wm; gc.collect(); torch.cuda.empty_cache()
wer_fp8=np.mean(aw) if aw else 1.0; cer_fp8=np.mean(ac) if ac else 1.0
print(f"WER FP8: {wer_fp8:.4f} | CER FP8: {cer_fp8:.4f}")


---
# Sprint 8: Benchmark Final (6 Modelos + Estadistica)

> 200 muestras del split test de CV17.
> Metricas: VRAM, RTF, WER, CER, PESQ, MCD, Disco.
> Shapiro-Wilk + Wilcoxon + Graficos + Tabla LaTeX.


In [ ]:
# === Sprint 8: Benchmark Final (placeholder - se completa al tener todos los modelos) ===
print("Sprint 8 se ejecuta despues de generar los 6 modelos.")
print("Modelos: FP16, RTN, GPTQ, AWQ, INT8, NF4, FP8")
print("200 muestras del split test de CV17")
print("Metricas: VRAM, RTF, WER, CER, PESQ, MCD, Disco")
print("Stats: Shapiro-Wilk + Wilcoxon signed-rank")
print("Output: outputs/sprint8/ (CSV, graficos PNG, tabla LaTeX)")
